# download-aic — tải zip video giao thông, unzip tại chỗ, xóa zip

Mỗi shard xử lý các file có chỉ số trong `[tu, den)` (thường 1 file ~10 GB):
tải zip về `/kaggle/working/out`, bung vào `OUT/<tên>/`, xóa zip để output
chỉ còn video + manifest. Ghi một dòng manifest mỗi file xong, rồi ghi mốc
`.moc.json` cho CLI ghép. Dòng manifest chỉ ghi khi thư mục đã bung xong
nên chạy dở chạy lại là tự nối tiếp (unzip đè bằng `-o`).

In [ ]:
# ORCHESTRA-PARAMS — kaggle_orchestra.py thay cell này lúc đẩy. Giá trị dưới chỉ để chạy tay.
SHARD = {"job": "tay", "chi_so": 0, "so_shard": 1, "tu": 0, "den": 1, "commit": "", "noi_tiep": None, "output": "manifest.jsonl", "base": "https://aic-data.ledo.io.vn", "files": ["Video_N001-N010"]}

In [ ]:
import json, shutil, subprocess
from pathlib import Path

OUT = Path("/kaggle/working/out"); OUT.mkdir(parents=True, exist_ok=True)
ra = OUT / SHARD["output"]

# Nối tiếp: chép manifest cũ từ dataset orch-<job>-s<i> của acc trước; thư mục video
# đã bung nếu có cũng chép sang để khỏi tải lại.
if SHARD.get("noi_tiep"):
    for f in Path("/kaggle/input").glob("**/*.jsonl"):
        if SHARD["noi_tiep"] in str(f):
            shutil.copy(f, OUT / f.name)
            print("noi tiep manifest:", f)
    for d in Path("/kaggle/input").glob(f"**/{SHARD['noi_tiep']}/*"):
        if d.is_dir() and not (OUT / d.name).exists():
            shutil.copytree(d, OUT / d.name)
            print("noi tiep dir:", d.name)

def done_ids():
    ids = set()
    if ra.exists():
        for line in ra.open(encoding="utf-8"):
            try:
                ids.add(json.loads(line)["id"])
            except Exception:
                pass
    return ids

def dir_xong(d):
    return d.is_dir() and any(d.iterdir())

xong = done_ids()
with ra.open("a", encoding="utf-8") as f:
    for k in range(SHARD["tu"], SHARD["den"]):
        idk = f"n{k:03d}"
        name = SHARD["files"][k]
        d = OUT / name
        if idk in xong and dir_xong(d):
            continue
        dst = OUT / f"{name}.zip"
        if not dir_xong(d):
            r = subprocess.run(["curl", "-L", "--fail", "--retry", "5",
                                  "--retry-all-errors", "-C", "-", "-o", str(dst),
                                  f"{SHARD['base']}/{name}.zip"])
            if r.returncode != 0:
                # ponytail: ConnectionError nam trong MANG_RE cua CLI -> loi mang tu doi acc,
                # khong danh Hong ca luot nhu AssertionError.
                raise ConnectionError(f"curl {name} exit {r.returncode} — mang")
            assert dst.stat().st_size > 0, f"{name} rong"
            free = shutil.disk_usage(OUT).free
            assert free > dst.stat().st_size, f"het dia: free {free} < zip {dst.stat().st_size}"
            r = subprocess.run(["unzip", "-q", "-o", str(dst), "-d", str(d)])
            assert r.returncode == 0, f"unzip {name} exit {r.returncode}"
            dst.unlink()  # ponytail: xoa zip giu output ~10GB thay vi ~21GB
        files = [p for p in d.rglob("*") if p.is_file()]
        n = sum(p.stat().st_size for p in files)
        assert files, f"{name} bung ra rong"
        f.write(json.dumps({"id": idk, "dir": name, "files": len(files), "bytes": n}) + "\n")
        f.flush()
        print("xong", name, len(files), n)

# ponytail: moc inline thay vì import ghi_moc từ src — job không cần dataset src nào.
moc = {"tu": SHARD["tu"], "den": SHARD["den"], "so_dong": len(done_ids()), "xong": True}
Path(str(ra) + ".moc.json").write_text(json.dumps(moc))
print(moc)
